# Reddit Logs to YTMusic Playlists

## Setup

In [1]:
YEAR=2024

# Set Paths

# Path to header .json file for ytmusic api
ytmusic_header_path = '..\\..\\oauth.json'
# Path to .tsv reddit search logs filtered to just new entries
reddit_log_path = '..\\..\\..\\reddit-scraper\\db'
search_db_path = '..\\..\\..\\reddit-scraper\\logs'

manual_labels_file = 'reddit_all_manual_labels.tsv'

## Imports / Helpers

In [2]:
import os
import glob
import time

import unicodedata
from datetime import date
import re
import pandas as pd
from IPython.display import display
from fuzzywuzzy import fuzz
from ytmusicapi import YTMusic

from datetime import date


### DataFrame Helpers

In [3]:

match_tsv_col_order = ['manual_label', 'ytmusic_key', 'reddit_title', 'match_quality',
       'match_score_token_set_ratio', 'match_score_token_sort_ratio',
       'is_album', 'reddit_sub', 'ytmusic_album', 'ytmusic_albumId',
       'ytmusic_artist', 'ytmusic_artistId', 'ytmusic_title',
       'ytmusic_videoId', 'ytmusic_duration', 'ytmusic_year',
       'ytmusic_resultType', 'reddit_key', 'reddit_post_id', 'reddit_sub_id',
       'reddit_aggregator', 'reddit_source_url', 'youtube_videoId']


def save_df_to_tsv(df, base_file_name, search_db_path):
    """
    Save the given DataFrame to a TSV file in the specified path with a name including the current date.
    """
    today_str = date.today().strftime('%Y-%m-%d')
    file_name = f'{base_file_name}_{today_str}.tsv'
    full_path = os.path.join(search_db_path, 'ytmusic', file_name)
    df.to_csv(full_path, sep='\t', header=True)
    print(f'Successfully saved DataFrame with shape: {df.shape} to {full_path}')


def load_tsv_to_df(file_name, search_db_path):
    """
    Load a TSV file into a DataFrame, add a 'reddit_post_id' column, and return the DataFrame and frozenset of IDs.
    """
    db_tsv_path = os.path.join(search_db_path, 'ytmusic', file_name)
    db = pd.read_csv(db_tsv_path, sep='\t', index_col=0)
    db['reddit_post_id'] = db.reddit_sub + '//' + db.reddit_source_url
    print(f'Loaded {len(db)} entries from {db_tsv_path}')
    return db

def remove_duplicates(df, key='reddit_post_id', keep='first', check_inconsitent_manual_label=False):
    """
    Remove duplicate rows based on the specified key and check for 'manual_label' inconsistencies.
    """
    if key not in df.columns:
        print(f"Warning: Key '{key}' not found in DataFrame. No duplicates removed.")
        return df
    
    if check_inconsitent_manual_label:
        # Check for inconsistencies in 'manual_label' for duplicates
        duplicates = df[df.duplicated(key, keep=keep)]
        inconsistent = duplicates.groupby(key).filter(lambda x: x['manual_label'].nunique() > 1)
        if not inconsistent.empty:
            print(f"Warning: Inconsistent 'manual_label' values found for some keys: {inconsistent[key].unique()}")

    # Remove duplicates and print the number of duplicates being removed
    before_removal = len(df)
    df = df.drop_duplicates(subset=key)
    after_removal = len(df)
    print(f'Removed {before_removal - after_removal} duplicate entries based on {key}.')

    return df


### YTMusic API and Functions

In [4]:
def parse_ytmusic_tracks(track_list):
    tracks = pd.DataFrame(track_list)
    tracks['artistId'] = tracks['artists'].dropna().apply(lambda x: x[0]['id'])
    tracks['artist'] = tracks['artists'].dropna().apply(lambda x: x[0]['name'])
    tracks['albumId'] = tracks['album'].dropna().apply(lambda x: x['id'])
    tracks['album'] = tracks['album'].dropna().apply(lambda x: x['name'])
    tracks = tracks.drop('thumbnails', axis=1)
    tracks = tracks.drop('artists', axis=1)
    return tracks

def parse_ytmusic_playlist(yt, playlist_meta):
    playlist_meta.pop('thumbnails', None)
    track_list = playlist_meta.pop('tracks', None)
    tracks = parse_ytmusic_tracks(track_list)
    return tracks, playlist_meta
    
ytm = YTMusic(ytmusic_header_path)
yt_res_cache = {}
yt_unmatched_cache = {}

### String Helper Functions

In [5]:
def scrub_title(title):
    orig_title = title
    title = str(title).lower().strip()
    title = unicodedata.normalize('NFKD', title).encode('ascii', 'ignore').decode()
    title = title.replace('ft.', 'feat.')
    title = title.replace('| ', '(')
    # remove stuff at end of title
    for k in [
         'official video', 'music video', 'live video','lyric video', 'cover)', 'video)', 'prod.', 'produced by', 
         'album stream', 'album review', 'album version', 'full album','produced by', 'npr music tiny desk concert',
         'anniversary expanded edition']:
        if k in title:
            new_t = title.split(k)[0]
            if len(new_t) > 5:
                title = title.split(k)[0]
    start_char = ['[', '('] 
    for k in [
        'official', 'unoffical', 'free', 'explicit', 'video', 'music', 'nsfw', 'original', 'lyric', 'studio', 'vinyl',
        'full', 'album)', 'audio)', 'cover)', 'convert', 'thissongissick', 'duploc', 'prod', 'leak', 'from', 'lofi hip', 
        'remaster', 'uncensored', '720p', '1080p', '320k' 'repackag', '19', '20', 'dir', 'quality upgrade', 'complete', 
        'with lyrics', 'visualizer', 'deluxe', 'anniversary edition']:
        for s in start_char:
            t = s + k
            if t in title:
                new_t = title.split(t)[0]
                if len(new_t) > 5:
                    title = title.split(t)[0]
    for s in start_char:
        if title.endswith(s):
            title = title[:-1]     
    # remove tokens from title
    for k in ['[hd]', '[hq]', 'hd', 'hq', '()', '[]', ' | ', '{}']:
        if k in title:
            title=title.replace(k, '')
    if title.endswith(' - '):
        title = title[0:-3]
    return title
    

def check_album_scrub_title(title):
    # check for album
    title = title.lower().strip()
    is_album = False
    for k in ['full album', '(album)', 'album stream']:
        if k in title:
            is_album = True

    title = scrub_title(title)
    return title, is_album

def strip_non_alphanumeric(value):
    value = str(re.sub('[^\\w\\s-]', ' ', str(value)))
    value = str(re.sub('[-\\s]+', ' ', str(value)))
    return value.strip()    

def extract_match_scores(query, match):
    q = strip_non_alphanumeric(query).lower()
    m = strip_non_alphanumeric(match).lower()
    return {
        'token_set_ratio': fuzz.token_set_ratio(q, m),
        # 'ratio': fuzz.ratio(q, m),
        'token_sort_ratio': fuzz.token_sort_ratio(q, m),
    }

## Parse Reddit .tsv and query YTMusic for Match

* Now checks db tsv to see if ialready matched (basd on sub and url)
* Some reddit entries are albums, most are tracks
* lots of title 'scrubbing' before query to clean
* saves match and unmatched seperatly, caches query responses 
    * cache in above ytmusic api cell
* scores match using fuzz metrics
    * tries to automate passing macthes with score threshold


#### Last run 1/1/2025




In [6]:
db = load_tsv_to_df(manual_labels_file, search_db_path)
prev_match_ids = frozenset(db['reddit_post_id'])
print(f'{prev_match_ids} previous entries in set')

C:\Users\jake\AppData\Local\Temp\ipykernel_7352\1411143118.py:26: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  db = pd.read_csv(db_tsv_path, sep='\t', index_col=0)


Loaded 99087 entries from ..\..\..\reddit-scraper\logs\ytmusic\reddit_all_manual_labels.tsv
frozenset({'chillmusic//https://youtube.com/watch?v=URvbvPp5Okc&feature=share', 'witch-house//https://www.youtube.com/watch?v=Hs5dAZSMCPc', 'IndieFolk//https://www.youtube.com/watch?v=3-sDIWLnOO8&list=OLAK5uy_mxcmDNkCO5-57GfkXN91KNsxqgtQOBCT8', 'neopsychedelia//https://www.youtube.com/watch?v=fye1XtXQn9s', 'hiphop//http://www.youtube.com/watch?v=ZrlJX7DzLhI&feature=channel_video_title', 'futuresynth//https://soundcloud.com/kgrime/01-dark-disco-mix/comment-1625921833', 'funkhouse//https://www.youtube.com/watch?feature=player_embedded&v=mgjU6M9Cmco', 'jazzyhiphop//http://www.youtube.com/watch?v=QlDUsVW5jVQ', 'chillmusic//https://www.youtube.com/watch?v=SMSeD7IxvXo', 'nudisco//https://soundcloud.com/robotaki/ghostboy-feat-claire-ridgely', 'theOverload//https://www.youtube.com/watch?v=dbjUKNC7dEM', 'VintageObscura//https://www.youtube.com/watch?v=cxrki6vu6Ks&list=WL&index=3', 'treemusic//https://www

In [ ]:
reddit_subfolders = [f'new_{YEAR}']
log_every_n_matches = 1000
tail_n_entries = 10
fname_splitter = '_'
expected_fname_toks = 2


log_tsvs = []
for folder in reddit_subfolders:
    tsv_path = os.path.join(reddit_log_path, folder)
    log_tsvs += list(glob.glob(os.path.join(tsv_path, '*.tsv')))
log_tsvs = sorted(log_tsvs)
print(f'Found {len(log_tsvs)} reddit tsvs')


matched_entries = []
unmatched_entries = []
t0 = time.time()
expected_cols = ['reddit', 'youtube_id', 'title', 'url', 'author', 'timestamp',
                 'description', 'likes', 'dislikes', 'num_comments', 'num_plays',
                 'is_media', 'thumb_url', 'num_views']
for i, tsv_file in enumerate(log_tsvs, start=1):
    name = os.path.splitext(os.path.basename(tsv_file))[0]
    toks = name.split(fname_splitter)
    if len(toks) != expected_fname_toks:
        print(
            f'skipping tsv without {expected_fname_toks} "{fname_splitter}" split toks: {tsv_file}')
        continue
    sub, agg = toks
    df = pd.read_csv(tsv_file, sep='\t', index_col=0)
    if len(df) == 0:
        print(f'\n\nSkipping empty tsv: {tsv_file}')
        continue
    print(f'({i}/{len(log_tsvs)})  Loaded {len(df)} entries with {len(df.columns)} columns from {name}')
    assert set(df.columns) == set(expected_cols)

    # Loop thru entries
    for entry in df.itertuples():
        if 'youtube' not in entry.url and 'soundcloud' not in entry.url:
            print(f' skipping, Unknown url source {entry.url}')
        title_key, is_album = check_album_scrub_title(entry.title)
        
        # If already in db get match from there
        post_id = f'{sub}//{entry.url}'
        if post_id in prev_match_ids:
            continue
        # Check cache for saved YTMusic query response or previous match failures        
        if entry.url in yt_res_cache:
            match = yt_res_cache[entry.url]
        elif title_key in yt_unmatched_cache:
            print(f'Skipping: previously unmatched entry: {title_key}')
            continue
        # Query YTMusic
        else:
            match = {}
            try:
                if is_album:
                    res = ytm.search(query=title_key, filter='albums', limit=1)
                    if len(res):
                        res = res[0]
                        match['is_album'] = True
                        match['ytmusic_title'] = ''
                        match['ytmusic_album'] = res.get('title', '')
                        match['ytmusic_albumId'] = res.get('browseId', '')
                        if 'artists' in res:
                            match['ytmusic_artist'] = res['artists'][0].get('name', '')
                            match['ytmusic_artistId'] = res['artists'][0].get('id', '')
                        match['ytmusic_key'] = f"{match.get('ytmusic_artist', '').lower()} - {scrub_title(match.get('ytmusic_album', ''))}"

                else:
                    res = ytm.search(query=title_key, filter='songs', limit=1)
                    if len(res):
                        res = res[0]
                        match['is_album'] = False
                        match['ytmusic_album'] = ''
                        if 'album' in res:
                            match['ytmusic_album'] = res['album'].get('name', '')
                            match['ytmusic_albumId'] = res['album'].get('id', '')
                        if 'artists' in res:
                            match['ytmusic_artist'] = res['artists'][0].get('name', '')
                            match['ytmusic_artistId'] = res['artists'][0].get('id', '')
                        match['ytmusic_title'] = res.get('title', '')
                        match['ytmusic_videoId'] = res.get('videoId', '')
                        match['ytmusic_key'] = f"{match.get('ytmusic_artist', '').lower()} - {scrub_title(match.get('ytmusic_title', ''))}"

            except Exception as e:
                print(f'Error with {entry.title}: {e}')
                pass

            # Store unmatched ytmusic query
            if len(res) == 0:
                unmatch = {}
                unmatch['reddit_title'] = entry.title
                unmatch['reddit_source_url'] = entry.url
                unmatch['reddit_key'] = title_key
                unmatch['reddit_sub'] = sub
                unmatch['reddit_post_id'] = post_id
                unmatch['reddit_sub_id'] = f'{sub}//{entry.url}'
                unmatch['reddit_aggregator'] = agg
                unmatch['manual_label'] = 'no-match'
                unmatched_entries.append(unmatch)
                yt_unmatched_cache[title_key] = unmatch
                print(f'Skipping : {entry.title}')
                continue


            # Other ytmusic fields
            # match['ytmusic_entry'] = f"{match.get('ytmusic_artist', '')} - {match.get('ytmusic_title', '')} - {match.get('ytmusic_album', '')}"
            match['ytmusic_duration'] = res['duration']
            match['ytmusic_year'] = res['year']
            match['ytmusic_resultType'] = res['resultType']
            yt_res_cache[entry.url] = match

        # Reddit fields
        match['reddit_title'] = entry.title
        match['reddit_title_len'] = len(str(entry.title))
        match['reddit_key'] = title_key
        match['reddit_sub'] = sub
        match['reddit_post_id'] = post_id
        match['reddit_sub_id'] = f'{sub}//{entry.url}'
        match['reddit_aggregator'] = agg
        match['reddit_source_url'] = entry.url
        match['youtube_videoId'] = entry.youtube_id
        match['manual_label'] = f'no-label_{date.today()}'

        # Match Score
        try:
            scores = extract_match_scores(
                match['reddit_key'], match['ytmusic_key'])
            for sk, sv in scores.items():
                match[f'match_score_{sk}'] = sv
            # Using manual grading to set thresholds
            if scores['token_sort_ratio'] < 40 or scores['token_set_ratio'] < 60:
                match['match_quality'] = 0.
            elif scores['token_sort_ratio'] > 75 or scores['token_set_ratio'] > 75:
                match['match_quality'] = 1.
            else:
                match['match_quality'] = 0.5
        except Exception as e:
            match['match_quality'] = 0.0
            print(f'Skipping match score for {entry.title}.\n  Match Error: {e}')


        # Update match results
        matched_entries.append(match)
print(f'\nFinished matching in {time.time() - t0 // 60:0.1f} minutes')
# Process Matched entries
match_df = pd.DataFrame(matched_entries)[match_tsv_col_order]
match_df = remove_duplicates(match_df, key='reddit_sub_id', keep='first')
save_df_to_tsv(match_df,  f'reddit_2024-new_ytmusic_scored_new_matches.tsv', search_db_path)

# Process Unmatched entries
unmatch_df = pd.DataFrame(unmatched_entries)
unmatch_df = remove_duplicates(unmatch_df, key='reddit_sub_id', keep='first')
save_df_to_tsv(unmatch_df,  f'reddit_2024-new_ytmusic_failed_new_matches.tsv', search_db_path)

Found 105 reddit tsvs
(1/105)  Loaded 501 entries with 14 columns from 2000smusic_all


In [22]:
new_f = 'reddit_2024-new_ytmusic_scored_new_matches.tsv_2024-12-31.tsv'
new_matches =  load_tsv_to_df(new_f, search_db_path)

match_f = 'reddit_all_manual_labels.tsv'
graded_matches =  load_tsv_to_df(match_f, search_db_path)

print(f"graded_matches shape {graded_matches.shape}")
print(f"new_matches shape {new_matches.shape}")

new_matches_filtered = new_matches[
    ~new_matches["reddit_title"].isin(graded_matches["reddit_title"])
]

print(
    f"new_matches_filtered shape after removing duplicates: {new_matches_filtered.shape}"
)

print(new_matches_filtered['match_quality'].value_counts())

Loaded 16664 entries from ..\..\..\reddit-scraper\logs\ytmusic\reddit_2024-new_ytmusic_scored_new_matches.tsv_2024-12-31.tsv


C:\Users\jake\AppData\Local\Temp\ipykernel_15756\1411143118.py:26: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  db = pd.read_csv(db_tsv_path, sep='\t', index_col=0)


Loaded 84045 entries from ..\..\..\reddit-scraper\logs\ytmusic\reddit_all_manual_labels.tsv
graded_matches shape (84045, 23)
new_matches shape (16664, 23)
new_matches_filtered shape after removing duplicates: (15664, 23)


## Manually grade using gsheet
https://docs.google.com/spreadsheets/d/1Z6X6rmOoLTqo3na_8Owrdf2O6FJ08n71buPhiP5uawA/edit#gid=2087986722


In [7]:
graded_tsv_file = 'reddit_2024-new_ytmusic_scored_new_matches__model-graded_2024-1-5.tsv'


# TODO try this graded_matches = load_tsv_to_df(graded_tsv_file, search_db_path)
graded_matches =  load_tsv_to_df(graded_tsv_file, search_db_path)
# graded_matches = pd.read_csv(os.path.join(search_db_path, 'ytmusic', graded_tsv_file), sep='\t')
graded_matches = graded_matches.sort_values('reddit_sub')

passing = graded_matches.loc[graded_matches.manual_label == 'pass']
failing = graded_matches.loc[graded_matches.manual_label == 'fail']
print(f'{graded_matches.shape} shaped graded matches, {len(passing)} passing, {len(failing)} failing')




Loaded 15042 entries from ..\..\..\reddit-scraper\logs\ytmusic\reddit_2024-new_ytmusic_scored_new_matches__model-graded_2024-1-5.tsv
(15042, 25) shaped graded matches, 10284 passing, 4758 failing


In [8]:
# # Do this once
# all_manual_labels = load_tsv_to_df(manual_labels_file, search_db_path)
# manual_labels = pd.concat([
#   graded_matches[all_manual_labels.columns], 
#   all_manual_labels
# ]).sort_values(['manual_label','ytmusic_key'], ascending=False)

# Replace old manual labels
# print(f'Loaded history with shape: {manual_labels.shape}')
# manual_labels = remove_duplicates(manual_labels, check_inconsitent_manual_label=True)
# print(f'Loaded and de-duped history, now has shape: {manual_labels.shape} and manual labels:\n{manual_labels.manual_label.value_counts()}')
# save_df_to_tsv(manual_labels,  f'{manual_labels_file}_new_.tsv', search_db_path)
# print(f'Manually overwrite _new over {manual_labels_file}')


## Make Playlists

In [30]:
import os
import sys
path_backup = os.path.join('../../')
module_path = os.path.abspath(path_backup)
if module_path not in sys.path:
    sys.path.append(module_path)
from ytmusic_library import YTMusicPlaylists
import ytmusicapi as ytmusicapi
print(f'Using ytmusicapi version: {ytmusicapi.__version__}')


RUN_API_AUTH_TEST = True
HEADER_FILE = path_backup + 'oauth.json'
PLAYLIST_TSV_DIR = path_backup + 'playlists/'

PLAYCOUNT_FILE='_ytmusic_lastfm_playcount.tsv'
NOT_LIKE_PLAYLIST_TSV ='_not_liked_tracks.tsv'


Y = YTMusicPlaylists(header=HEADER_FILE, playlist_tsv_dir=PLAYLIST_TSV_DIR)
if RUN_API_AUTH_TEST: Y.test_ytmusic_api()
print(f"Loaded {len(Y.playlists['title'].unique())} playlists")

passing_albums = passing.loc[passing['is_album'] == True]
# passing_tracks = passing.loc[passing.is_album == False]
passing_tracks = passing.loc[passing['is_album'] == False] # may be str may be bool


Using ytmusicapi version: 1.8.2
Using header file: ../../oauth.json
Test Passed in 2.31 seconds
Using ytmusicapi version: 1.8.2
Loaded 511 playlists


## Subreddit playlists for passing tracks

(same code for round 1 and 2, just clear completed [])

will update if playlist exists, otherwise create a new radio for subreddit

In [10]:

# Use this if it gets stuck to get completed back to where it was
completed = []
# stop_sub = 'idm'
# n_subs = len(passing_tracks['reddit_sub'].unique())
# for sub, df in passing_tracks.groupby('reddit_sub'):

#   completed.append(sub)
#   if sub == stop_sub:
#     print(f'Stopped at {stop_sub}')
#     break

# print(f'So far completed {len(completed)} of {n_subs} ({len(completed)/n_subs:0.1%}%)')


In [32]:
# when you get banned it seem like it is for around 25 playlists, ban last 6-8 hrs?
# SLEEP_TIME=6*60*60
# print(f'Sleeping {SLEEP_TIME//60//60}hrs')
# time.sleep(SLEEP_TIME)
# 

SLEEP_TIME=20
LIMIT = 4000
MIN_N_LIKE = 5
DRY=False
# completed = []
# Note: r/song doesnt work probably too big? > 1000 tracks? maybe not
for sub, df in passing_tracks.groupby('reddit_sub'):
    if sub in completed:
        print(f'Already completed: {sub}')
        continue

    # Create YTMusic Playlist from subreddit entries
    vids = df.ytmusic_videoId.unique().tolist()
    vids = list(frozenset(vids) - Y.banned_vid_set)
    title=f'xr {sub} radio'
    print(f'\nGenerating {title} ytmusic playlist for {len(df)} passing tracks')


    if title in Y.playlists['title'].unique():
        # Update existing
        pl_id = Y.query_by_title(title).playlistId
        if not DRY:
          status = ytm.add_playlist_items(playlistId=pl_id, videoIds=vids, duplicates=False)
        print(f'Updated {len(vids)} tracks in {title} playlist with id: {pl_id}')
    else: # Create New
        desc = f'Matched {len(vids)} tracks in {title} playlist using filters: {df.reddit_aggregator.unique()}'
        pl_id=-1
        if not DRY:
          pl_id = ytm.create_playlist(title=title,  description=desc, privacy_status='PRIVATE', video_ids=vids)
        print(f'Saved {len(vids)} tracks to new {title} playlist with id: {pl_id}')
    print(f' Waiting {SLEEP_TIME} seconds...')
    time.sleep(SLEEP_TIME)


    # Fetch Just Saved Playlist
    Y.clean_up_radio_playlist(
        Y.playlist_get_info(pl_id), verbose=True, 
        move_like=True, min_num_like=MIN_N_LIKE,
        sleep=1, create_like_playlist=True, 
        remove_dislike=True, remove_not_like=True
    )
    
    Y.playcount_sort_playlist(Y.playlist_get_info(pl_id, use_cache=False), ignore_banned=True)
    completed.append(sub)
    


Already completed: 2000smusic
Already completed: 2010smusic
Already completed: 50sMusic
Already completed: 60sMusic
Already completed: 70s
Already completed: 70sMusic
Already completed: 80sHipHop
Already completed: 80sMusic
Already completed: 90sAlternative
Already completed: 90sMusic
Already completed: 90sRock
Already completed: 90shiphop
Already completed: AfricanMusic
Already completed: AtmosphericDnB
Already completed: ClassicRock
Already completed: DoomMetal
Already completed: DreamPop
Already completed: ElectronicMusic
Already completed: Elephant6
Already completed: Exotica
Already completed: Flamenco
Already completed: GypsyJazz
Already completed: IndieFolk
Already completed: Instrumentals
Already completed: MelancholyMusic
Already completed: ModernRockMusic
Already completed: NewWave
Already completed: OldElectronicMusic
Already completed: OldiesMusic
Already completed: OldskoolRave
Already completed: OutlawCountry
Already completed: PostRock
Already completed: PsychedelicRock


## Subreddit playlists for passing albums
(same code for round 1 and 2, just clear completed [])

In [33]:
completed= []
# completed = ['90shiphop', 'blues', 'chillmusic', 'chillwave', 'futurebass', 'futurebeats', 'futurefunkairlines', 'hiphop', 'hiphop101', 'indie', 'indieheads', 'indierock', 'jazz', 'jazzyhiphop', 'lofihiphop', 'psychedelicrock', 'rap', 'realdubstep', 'reggae', 'shoegaze', 'treemusic', 'triphop']
# stop_sub = 'idm'
# n_subs = len(passing_tracks['reddit_sub'].unique())
# for sub, df in passing_tracks.groupby('reddit_sub'):

#   completed.append(sub)
#   if sub == stop_sub:
#     print(f'Stopped at {stop_sub}')
#     break

# print(f'So far completed {len(completed)} of {n_subs} ({len(completed)/n_subs:0.1%}%)')


In [35]:
SLEEP_TIME=20
LIMIT = 900
MIN_N_LIKE = 5

for sub, df in passing_albums.groupby('reddit_sub'):
    if sub in completed:
        print(f'Already completed: {sub}')
        continue

    # Create YTMusic Playlist from subreddit entries
    print(f'\n{title} ytmusic playlist for {len(df)} passing albums')
    vids = set()
    for row in df.itertuples():
        print(f' Adding album: {row.ytmusic_album}')
        try:
            for track in  ytm.get_album(row.ytmusic_albumId).get('tracks', []):
                vids.add(track['videoId'])
        except Exception as e:
            print(row, sub, e)
    vids = list(vids)
        
    title=f'xr {sub} albums'
    if title in Y.playlists['title'].unique():
        # Update existing
        pl_id = Y.query_by_title(title).playlistId
        status = ytm.add_playlist_items(playlistId=pl_id, videoIds=vids, duplicates=False)
        print(f'Updated {len(vids)} {title} albums playlist with id: {pl_id}, waiting {SLEEP_TIME} seconds...')
    else: # Create New
        desc = f'Matched {len(vids)} albums from {title} using filters: {df.reddit_aggregator.unique()}'
        pl_id = ytm.create_playlist(title=title,  description=desc, privacy_status='PRIVATE', video_ids=vids)
        print(f'Saved {len(vids)} {title} albums playlist with id: {pl_id}, waiting {SLEEP_TIME} seconds...')
    time.sleep(SLEEP_TIME)

    # Fetch Just Saved Playlist
    metadata = ytm.get_playlist(pl_id, limit=LIMIT)
    tracks, metadata = parse_ytmusic_playlist(ytm, metadata)

    # Create Like subset to merge with tracks playlist
    liked_tracks = tracks.loc[tracks['likeStatus'] == 'LIKE']
    if len(liked_tracks) < MIN_N_LIKE:
        print(f'Not enough LIKE tracks to split into new playlists: count = {len(liked_tracks)}')
        continue
    vids = liked_tracks.videoId.unique().tolist()
    title=f'xr {sub}'
    if title in Y.playlists['title'].unique():
        # Update existing
        pl_id = Y.query_by_title(title).playlistId
        status = ytm.add_playlist_items(playlistId=pl_id, videoIds=vids, duplicates=False)
        print(f'Updated {len(vids)} {title} like album tracks playlist with id: {pl_id}, waiting {SLEEP_TIME} seconds...')
    else: # Create New
        desc = f'Matched {len(vids)} like  albumtracks from {title} using filters: {df.reddit_aggregator.unique()}'
        pl_id = ytm.create_playlist(title=title,  description=desc, privacy_status='PRIVATE', video_ids=vids)
        print(f'Saved {len(vids)} {title} like album tracks playlist with id: {pl_id}, waiting {SLEEP_TIME} seconds...')
    time.sleep(SLEEP_TIME)
    # Fetch Just Saved (like) Playlist
    Y.clean_up_radio_playlist(
        Y.playlist_get_info(pl_id), verbose=True, 
        move_like=True, min_num_like=MIN_N_LIKE,
        sleep=1, create_like_playlist=True, 
        remove_dislike=True, remove_not_like=True
    )
    
    Y.playcount_sort_playlist(Y.playlist_get_info(pl_id, use_cache=False), ignore_banned=True)
    completed.append(sub)




xr acidhouse albums ytmusic playlist for 5 passing albums
 Adding album: Polynesia
 Adding album: Caribbean Moonlight
 Adding album: Bwana A
 Adding album: Voodoo!
 Adding album: Breeze From The East
Saved 61 xr Exotica albums albums playlist with id: PLWptjpDqazOxKVEoMtmguqYL64g4G7rwZ, waiting 20 seconds...
Not enough LIKE tracks to split into new playlists: count = 0

xr Exotica albums ytmusic playlist for 4 passing albums
 Adding album: Youth
 Adding album: True North
 Adding album: Heaven Is Gone
 Adding album: Destroy + Rebuild
Saved 46 xr ModernRockMusic albums albums playlist with id: PLWptjpDqazOySqyrn8l--3B_4_0ydiGAM, waiting 20 seconds...
Not enough LIKE tracks to split into new playlists: count = 0

xr ModernRockMusic albums ytmusic playlist for 2 passing albums
 Adding album: Look Again
 Adding album: This Is Big Audio Dynamite
Saved 10 xr NewWave albums albums playlist with id: PLWptjpDqazOwydemxv0-bgrDyW3IOSy7Y, waiting 20 seconds...
Not enough LIKE tracks to split into 